# Modeling with PyCaret: Road Accident Prediction

สมุดบันทึกนี้อ้างอิง feature, target และการบันทึก artifact ชุดเดียวกับ `src/model/train_pycaret.py` เพื่อให้ workflow ใน notebook สอดคล้องกับโมเดลที่ใช้ในแอปจริง

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError("ไม่พบโฟลเดอร์ src กรุณาเปิด notebook จาก project root หรือโฟลเดอร์ notebooks")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from pycaret.regression import compare_models, finalize_model, predict_model, pull, save_model, setup, tune_model
from src.data.feature_engineering import build_model_features
from src.model.evaluate import regression_metrics, save_metrics, save_predictions_sample
from src.model.train_pycaret import FEATURES, TARGET
from src.utils.config import (
    LEADERBOARD_CSV,
    METRICS_JSON,
    MODEL_BASENAME,
    PREDICTIONS_SAMPLE_CSV,
    PROCESSED_PARQUET,
    ensure_project_dirs,
)


In [ ]:
df = pd.read_parquet(PROCESSED_PARQUET)
df = build_model_features(df)

use_cols = [c for c in FEATURES if c in df.columns] + [TARGET]
df_model = df[use_cols].copy()
df_model[TARGET] = pd.to_numeric(df_model[TARGET], errors="coerce")
df_model = df_model.dropna(subset=[TARGET])

cat_cols = df_model.drop(columns=[TARGET]).select_dtypes(include=["object", "string", "category"]).columns.tolist()
num_cols = [c for c in df_model.columns if c not in cat_cols + [TARGET]]

for c in cat_cols:
    df_model[c] = df_model[c].astype("object")
    df_model[c] = df_model[c].replace({pd.NA: np.nan})
    df_model[c] = df_model[c].fillna("Unknown")
    df_model[c] = df_model[c].astype(str).str.strip()
    df_model.loc[df_model[c].isin(["", "nan", "None", "<NA>"]), c] = "Unknown"

for c in num_cols:
    df_model[c] = pd.to_numeric(df_model[c], errors="coerce")

df_model = df_model.dropna(subset=num_cols, how="any").reset_index(drop=True)

SAMPLE_SIZE = 3000  # เปลี่ยนเป็น None หากต้องการใช้ข้อมูลทั้งหมด
if SAMPLE_SIZE and len(df_model) > SAMPLE_SIZE:
    df_model = df_model.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print(f"training rows: {len(df_model):,}")
print(f"features: {FEATURES}")
df_model.head()


In [ ]:
ensure_project_dirs()

_ = setup(
    data=df_model,
    target=TARGET,
    session_id=42,
    fold=5,
    train_size=0.8,
    verbose=False,
    numeric_imputation="median",
    categorical_imputation="most_frequent",
)

best = compare_models()
leaderboard = pull()
leaderboard


In [ ]:
tuned = tune_model(best)
final_model = finalize_model(tuned)
save_model(final_model, str(MODEL_BASENAME))
leaderboard.to_csv(LEADERBOARD_CSV, index=False, encoding="utf-8-sig")

print(f"saved model: {MODEL_BASENAME}.pkl")
print(f"saved leaderboard: {LEADERBOARD_CSV}")


In [ ]:
pred_df = predict_model(final_model, data=df_model)
metrics = regression_metrics(pred_df[TARGET], pred_df["prediction_label"])
save_metrics(metrics, METRICS_JSON)
save_predictions_sample(pred_df, PREDICTIONS_SAMPLE_CSV)

print(f"saved metrics: {METRICS_JSON}")
print(f"saved predictions: {PREDICTIONS_SAMPLE_CSV}")
pd.DataFrame([metrics])


## หมายเหตุ

- notebook นี้ลดจำนวนแถวด้วย `SAMPLE_SIZE` เพื่อให้ทดลองได้เร็วขึ้นในงานสำรวจ
- หากต้องการสร้างโมเดล production เต็มชุด ให้ใช้ `python -m src.model.train_pycaret`